In [ ]:
#!pip install langchain langchain_openai langgraph langchain_community langchain_openai langchain_core ddgs -q

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph.message import add_messages
from dotenv import load_dotenv

from langgraph.prebuilt import ToolNode, tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool

import requests
import random

In [ ]:
# Configure OpenAI key if not already set
import os
os.environ['OPENAI_API_KEY'] = 'sk-xxxxxxxxxxxxxxxxxxxxxxx'

In [ ]:
llm = ChatOpenAI()

In [ ]:
# Tools
search_tool = DuckDuckGoSearchRun(region="us-en")

In [ ]:
@tool
def calculator(first_num: float, second_num: float, operation: str) -> dict:
    """
    Perform a basic arithmetic operation on two numbers.
    Supported operations: add, sub, mul, div
    """
    try:
        if operation == "add":
            result = first_num + second_num
        elif operation == "sub":
            result = first_num - second_num
        elif operation == "mul":
            result = first_num * second_num
        elif operation == "div":
            if second_num == 0:
                return {"error": "Division by zero is not allowed"}
            result = first_num / second_num
        else:
            return {"error": f"Unsupported operation '{operation}'"}

        return {"first_num": first_num, "second_num": second_num, "operation": operation, "result": result}
    except Exception as e:
        return {"error": str(e)}

- Create free API key : https://www.alphavantage.co/
- API key : xxxxxxxxxxxxxx

In [ ]:
@tool
def get_stock_price(symbol: str) -> dict:
    """
    Fetch latest stock price for a given symbol (e.g. 'AAPL', 'TSLA')
    using Alpha Vantage with API key in the URL.
    """
    url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={symbol}&apikey=xxxxxxxxxxxxxxxxx"
    r = requests.get(url)
    return r.json()

In [ ]:
# Make tool list
tools = [get_stock_price, search_tool, calculator]

# Make the LLM tool-aware
llm_with_tools = llm.bind_tools(tools)

In [ ]:
# state
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
# graph nodes
def chat_node(state: ChatState):
    """LLM node that may answer or request a tool call."""
    messages = state['messages']
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}



In [ ]:
tool_node = ToolNode(tools)  # Executes tool calls

In [ ]:
# graph structure
graph = StateGraph(ChatState)
graph.add_node("chat_node", chat_node)
graph.add_node("tools", tool_node)

In [ ]:
graph.add_edge(START, "chat_node")

# If the LLM asked for a tool, go to ToolNode; else finish
graph.add_conditional_edges("chat_node", tools_condition)

graph.add_edge("tools", "chat_node")

In [ ]:
chatbot = graph.compile()


In [ ]:

chatbot

In [ ]:
# Regular chat
out = chatbot.invoke({"messages": [HumanMessage(content="Hello!")]})

print(out["messages"][-1].content)

In [ ]:
# Chat requiring tool
out = chatbot.invoke({"messages": [HumanMessage(content="What is 2*3?")]})
print(out["messages"][-1].content)

In [ ]:
# Chat requiring tool
out = chatbot.invoke({"messages": [HumanMessage(content="What is the stock price of apple")]})
print(out["messages"][-1].content)

In [ ]:
# Chat requiring tool
out = chatbot.invoke({"messages": [HumanMessage(content="First find out the stock price of Apple using get stock price tool then use the calculator tool to find out how much will it take to purchase 50 shares?")]})
print(out["messages"][-1].content)

### TO get Serach Result - > Duck Duck Go

In [ ]:
# ---------------------------------------------------------
# Imports
# ---------------------------------------------------------
from langgraph.graph import StateGraph, END
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_openai import ChatOpenAI
from typing import TypedDict

In [ ]:
# ---------------------------------------------------------
# Define the state schema
# ---------------------------------------------------------
class NewsState(TypedDict):
    query: str  # User query (e.g., "latest AI news")
    search_results: str  # Raw search results from DuckDuckGo
    summary: str  # Summarized version of the results

In [ ]:
# ---------------------------------------------------------
# Define nodes (functions for the graph)
# ---------------------------------------------------------


def search(state: NewsState) -> NewsState:
    """Use DuckDuckGo search to get results for the query."""
    results = search_tool.invoke(state["query"])
    return {"search_results": results}


def summarize(state: NewsState) -> NewsState:
    """Summarize the search results using the LLM."""
    prompt = f"""
    Summarize the following news results into 3 short bullet points:

    {state['search_results']}
    """
    response = llm.invoke(prompt)
    return {"summary": response.content}

In [ ]:
# ---------------------------------------------------------
# Build the LangGraph
# ---------------------------------------------------------
graph = StateGraph(NewsState)

# Add nodes
graph.add_node("search", search)
graph.add_node("summarize", summarize)

# Define edges (flow of execution)
graph.set_entry_point("search")  # Start with search
graph.add_edge("search", "summarize")  # Then summarize
graph.add_edge("summarize", END)  # End of workflow

# Compile the graph
app = graph.compile()

In [ ]:
# ---------------------------------------------------------
# Run the pipeline
# ---------------------------------------------------------
query = "latest AI breakthroughs"
final_state = app.invoke({"query": query})

print("🔍 Query:", query)
print("\n📑 Summary:\n", final_state["summary"])